In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

path="NationalNames.csv"
data=pd.read_csv(path)

data["Name"]=data["Name"]
data=np.array(data["Name"][:10000]).reshape(-1,1)
data=[x.lower() for x in data[:,0]]
data=np.array(data).reshape(-1,1)

In [2]:
print("Data Shape= {}".format(data.shape))
print()
print("Lets see some names:")
print(data[1:10])

Data Shape= (10000, 1)

Lets see some names:
[['anna']
 ['emma']
 ['elizabeth']
 ['minnie']
 ['margaret']
 ['ida']
 ['alice']
 ['bertha']
 ['sarah']]


In [3]:
transform_data=np.copy(data)
maxLength=0
for index in range(len(data)):
    maxLength=max(maxLength,len(data[index,0]))

for index in range(len(data)):
    length=(maxLength-len(data[index,0]))
    string='.'*length
    transform_data[index,0]=''.join([transform_data[index,0],string])

In [4]:
print("Transformed Data")
print(transform_data[1:10])

Transformed Data
[['anna........']
 ['emma........']
 ['elizabeth...']
 ['minnie......']
 ['margaret....']
 ['ida.........']
 ['alice.......']
 ['bertha......']
 ['sarah.......']]


In [5]:
vocab=list()
for name in transform_data[:,0]:
    vocab.extend(list(name))

vocab=set(vocab)
vocabSize=len(vocab)
print("vocab size={}".format(len(vocab)))
print("vocab={}".format(vocab))

vocab size=27
vocab={'n', 'e', 'q', 'u', 'z', 'l', 'i', 'f', 'm', 'p', 'y', 'b', 'd', 'o', 'c', 'x', 'g', 't', 'h', 'a', 'r', 'k', 'j', '.', 'v', 'w', 's'}


In [6]:
charId=dict()
idChar=dict()
for i,char in enumerate(vocab):
    charId[char]=i
    idChar[i]=char

print('a-{},22-{}'.format(charId['a'],idChar[22]))

a-19,22-j


In [7]:
trainDataset=[]
batchSize=20
for i in range(len(transform_data)-batchSize+1):
    start =i*batchSize
    end=start+batchSize
    batchData=transform_data[start:end]
    if(len(batchData)!=batchSize):
        break


In [8]:
# Initialize variables
charList = []
batchSize = 20
trainDataset = []

# Iterate through the data to create batches
for i in range(len(transform_data) - batchSize + 1):
    start = i * batchSize
    end = start + batchSize
    batchData = transform_data[start:end]
    
    # Check if batchData is empty or not
    if len(batchData) == 0:
        print(f"Skipping empty batch at index {i}")
        continue
    
    # Check if each entry in batchData has the expected structure
    if len(batchData[0]) == 0 or len(batchData[0][0]) == 0:
        print(f"Skipping malformed data at index {i}")
        continue

    # Create the batch dataset for the current batch
    for k in range(len(batchData[0][0])):  # Iterate over the string length
        batchDataset = np.zeros([batchSize, len(vocab)])  # Initialize a new batch
        
        for j in range(batchSize):  # Iterate over the batch size
            name = batchData[j][0]
            if k < len(name):  # Ensure index is within the length of the string
                charIndex = charId[name[k]]  # Get the index for the character
                batchDataset[j, charIndex] = 1.0  # Set the corresponding position to 1.0
            else:
                print(f"Index {k} out of bounds for name: {name}")
        
        charList.append(batchDataset)
    
    # Add the current batch to the train dataset
    trainDataset.append(charList)
    charList = []  # Reset charList for the next batch

# Print the size of the training dataset
print(f"Total batches created: {len(trainDataset)}")


Skipping empty batch at index 500
Skipping empty batch at index 501
Skipping empty batch at index 502
Skipping empty batch at index 503
Skipping empty batch at index 504
Skipping empty batch at index 505
Skipping empty batch at index 506
Skipping empty batch at index 507
Skipping empty batch at index 508
Skipping empty batch at index 509
Skipping empty batch at index 510
Skipping empty batch at index 511
Skipping empty batch at index 512
Skipping empty batch at index 513
Skipping empty batch at index 514
Skipping empty batch at index 515
Skipping empty batch at index 516
Skipping empty batch at index 517
Skipping empty batch at index 518
Skipping empty batch at index 519
Skipping empty batch at index 520
Skipping empty batch at index 521
Skipping empty batch at index 522
Skipping empty batch at index 523
Skipping empty batch at index 524
Skipping empty batch at index 525
Skipping empty batch at index 526
Skipping empty batch at index 527
Skipping empty batch at index 528
Skipping empty

In [9]:
charList=[]
for k in range(len(batchData[0][0])):
    batchDataset=np.zeros([batchSize,len(vocab)])
    for j in range(batchSize):
        name=batchData[j][0]
        charIndex=charId[name[k]]
        batchDataset[j,charIndex]=1.0
    charList.append(batchDataset)
trainDataset.append(charList)

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
inputUnits=100
hiddenUnits=256
outputUnits=vocabSize
learningRate=0.005
beta1=0.90
beta2=0.99

In [ ]:
def sigmoid(X):
    return 1/(1+np.exp(-X))

def tanhActivation(X):
    return np.tanh(X)

def softmax(X):
    expX=np.exp(X)
    expXsum=np.sum(expX,axis=1).reshape(-1,1)
    expX=expX/expXsum
    return expX
def tanhDerivative(X):
    return 1-(X**2)

In [ ]:
def initializeParameters():
    mean=0
    std=0.01
    forgetGW=np.random.normal(mean,std,(inputUnits+hiddenUnits,hiddenUnits))
    inputGW=np.random.normal(mean,std,(inputUnits+hiddenUnits,hiddenUnits))
    outputGW=np.random.normal(mean,std,(inputUnits+hiddenUnits,hiddenUnits))
    gateGW=np.random.normal(mean,std,(inputUnits+hiddenUnits,hiddenUnits))
    hiddenOW=np.random.normal(mean,std,(hiddenUnits,outputUnits))
    parameters=dict()
    parameters['fgw']=forgetGW
    parameters['igw']=inputGW
    parameters['ogw']=outputGW
    parameters['ggw']=gateGW
    parameters['how']=hiddenOW
    return parameters 

In [ ]:
def lstmCell(batchDataset,prevActivationMatrix,prevCellMatrix,parameters):
    fgw=parameters['fgw']
    igw=parameters['igw']
    ogw=parameters['ogw']
    ggw=parameters['ggw']
    
    concatDataset=np.concatenate((batchDataset,prevActivationMatrix),axis=1)
    fa=np.matmul(concatDataset,fgw)
    fa=sigmoid(fa)
    
    ia=np.matmul(concatDataset,igw)
    ia=sigmoid(ia)
    
    oa=np.matmul(concatDataset,ogw)
    oa=sigmoid(oa)
    
    ga=np.matmul(concatDataset,ggw)
    ga=tanhActivation(ga)
    
    cellMemoryMatrix=np.multiply(fa,prevCellMatrix)+np.multiply(ia,ga)
    
    activationMatrix=np.multiply(oa,tanhActivation(cellMemoryMatrix))
    
    lstmActivation=dict()
    lstmActivation['fa']=fa
    lstmActivation['ia']=ia
    lstmActivation['oa']=oa
    lstmActivation['ga']=ga
    return lstmActivation,cellMemoryMatrix,activationMatrix

In [ ]:
def outputCell(activationMatrix,parameters):
    how=parameters['how']
    outputMatrix=np.matmul(activationMatrix,how)
    outputMatrix=softmax(outputMatrix)
    
    return outputMatrix

In [ ]:
def getEmbeddings(batchDataset,embeddings):
    embeddingDataset=np.matmul(batchDataset,embeddings)
    return embeddingDataset

In [10]:
def forwardPropagation(batches,parameters,embeddings):
    batchSize=batches[0].shape[0]
    
    lstmCache=dict()
    activationCache=dict()
    cellCache=dict()
    outputCache=dict()
    embeddingCache=dict()
    
    a0=np.zeros([batchSize,hiddenUnits],dtype=np.float32)
    c0=np.zeros([batchSize,hiddenUnits],dtype=np.float32)
    
    activationCache['a0']=a0
    cellCache['c0']=c0
    
    for i in range(len(batches)-1):
        batchDataset=batches[1]
        
        batchDataset=getEmbeddings(batchDataset,embeddings)
        embeddingCache['emb'+str(i)]=batchDataset
        
        lstmActivations,ct,at=lstmCell(batchDataset,a0,c0,parameters)
        ot=outputCell(at,parameters)
        
        lstmCache['lstm'+str(i+1)]=lstmActivations
        activationCache['a'+str(i+1)]=at
        cellCache['c'+str(i+1)]=ct
        outputCache['o'+str(i+1)]=ot
        a0=at
        c0=ct
    return embeddingCache,lstmCache,activationCache,cellCache,outputCache
        

In [11]:
def calLossAccuracy(batchLabels,outputCache):
    loss=0
    acc=0
    prob=1
    batchSize=batchLabels[0].shape[0]
    for i in range(1,len(outputCache)+1):
        labels=batchLabels[i]
        pred=outputCache['o'+str(i)]
        prob=np.multiply(prob,np.sum(np.multiply(labels,pred),axis=1).reshape(-1,1))
        loss+=np.sum(np.multiply(labels,np.log(pred)+np.multiply(1-labels,np.log(1-pred))),axis=1).reshape(-1,1)
        acc+=np.array(np.argmax(labels,1)==np.argmax(pred,1),dtype=np.float32).reshape(-1,1)
        perplexity=np.sum(1/prob)**(1/len(outputCache))/batchSize
        loss=np.sum(loss)*(-1/batchSize)
        acc=np.sum(acc)/(batchSize)
        acc=acc/len(outputCache)
        return perplexity,loss,acc

In [12]:
def calculateOutputCellError(batchLabels,outputCache,parameters):
    outputErrorCache=dict()
    activationErrorCache=dict()
    how=parameters['how']
    for i in range(1,len(outputCache)+1):
        labels=batchLabels[i]
        pred=outputCache['o'+str(i)]
        errorOutput=pred-labels
        errorActivation=np.matmul(errorOutput,how.T)
        outputErrorCache['eo'+str(i)]=errorOutput
        activationErrorCache['ea'+str(i)]=errorActivation
    return outputErrorCache,activationErrorCache

In [13]:
def calculateSingleLstmCellError(activationOutputError,nextActivationError,nextCellError,parameters,lstmActivation,cellActivation,prevCellActivation):
    activationError=activationOutputError+nextActivationError
    oa=lstmActivation['oa']
    eo=np.multiply(activationError,tanhActivation(cellActivation))
    eo=np.multiply(np.multiply(eo,oa),1-oa)
    cellError=np.multiply(activationError,oa)
    cellError=np.multiply(cellError,tanhDerivative(tanhActivation(cellActivation)))
    cellError+=nextCellError
    ia=lstmActivation['ia']
    ga=lstmActivation['ga']
    ei=np.multiply(cellError,ga)
    ei=np.multiply(np.multiply(ei,ia),1-ia)
    
    eg=np.multiply(cellError,ia)
    eg=np.multiply(eg,tanhDerivative(ga))
    
    fa=lstmActivation['fa']
    ef=np.multiply(cellError,prevCellActivation)
    ef=np.multiply(np.multiply(ef,fa),1-fa)
    prevCellError=np.multiply(cellError,fa)
    
    fgw=parameters['fgw']
    igw=parameters['igw']
    ggw=parameters['ggw']
    ogw=parameters['ogw']
    
    embedActivationError=np.matmul(ef,fgw.T)
    embedActivationError+=np.matmul(ei,igw.T)
    embedActivationError+=np.matmul(eo,ogw.T)
    embedActivationError+=np.matmul(eg,ggw.T)
    inputHiddenUnits=fgw.shape[0]
    hiddenUnits=fgw.shape[1]
    inputUnits=inputHiddenUnits-hiddenUnits
    prevActivationError=embedActivatonError[:,inputUnits:]
    embedError=embedActivationError[:,:inputUnits]
    lstmError=dict()
    lstmError['ef']=ef
    lstmError['ei']=ei
    lstmError['eo']=eo
    lstmError['eg']=eg
    return prevActivationError,prevCellError,embedError,lstmError
    

In [14]:
def calculateOutputCellDerivatives(outputErrorCache,activationCache,parameters):
    dhow=np.zeros(parameters['how'].shape)
    batchSize=activationCache['a1'].shape[0]
    for i in range(1,len(outputErrorCache)+1):
        outputError=outputErrorCache['eo'+str(i)]
        activation=activationCache['a'+str(i)]
        dhow+=np.matmul(activation.T,outputError)/batchSize
    return dhow

In [15]:
def calculateSingleLstmCellDerivatives(lstmError,EmbeddingMatrix,activationMatrix):
    ef=lstmError['ef']
    ei=lstmError['ei']
    eo=lstmError['eo']
    eg=lstmError['eg']
    concatMatrix=np.concatenate((embeddingMatrix,activationMatrix),axis=1)
    batchSize=embeddingMatrix.shape[0]
    dfgw=np.matmul(concatMatrix.T,ef)/batchSize
    digw=np.matmul(concatMatrix.T,ei)/batchSize
    dogw=np.matmul(concatMatrix.T,eo)/batchSize
    dggw=np.matmul(concatMatrix.T,eg)/batchSize
    derivatives=dict()
    derivatives['dfgw']=dfgw
    derivatives['digw']=digw
    derivatives['dogw']=dogw
    derivatives['dggw']=dggw
    return derivatives
    
    